<a href="https://colab.research.google.com/github/Praharshita1275/Agentic-Medical-Fact-Verification-system/blob/main/MEDICAL_CLAIM_VERIFIER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏥 Medical Claim Verifier — Ollama Cloud API Edition

| Feature | Description |
|---------|-------------|
| **E2** | Multi-Round Rebuttal Debate |
| **E3** | Live PubMed Search (NCBI Entrez) |
| **E4** | Uncertainty Quantification |
| **LLM** | `gemma3:27b` via **Ollama Cloud API** (`https://api.ollama.com`) |

**Team:** G Akshatha · Gaali Sai Praharshita · Srichandana  
**Dept of IT, CBIT Hyderabad**

---

### Setup checklist
1. Get a free Ollama Cloud API key → [ollama.com](https://ollama.com) → Sign in → API Keys
2. Paste the key in **Cell 1** where it says `OLLAMA_API_KEY = "..."`
3. Run all cells top to bottom.

> No local Ollama install needed. All inference runs on Ollama's cloud.


## Cell 1 — Install Dependencies

In [ ]:
!pip install -q openai sentence-transformers faiss-cpu \
               numpy pandas tqdm scikit-learn biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.3 MB/s eta 0:00:00


## Cell 2 — Configuration & Ollama Cloud Client

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Ollama Cloud API setup
#
# Ollama Cloud exposes an OpenAI-compatible endpoint:
#   Base URL : https://ollama.com/v1
#   Auth     : Bearer <your_api_key>
#   Docs     : https://ollama.com/docs/api
# ═══════════════════════════════════════════════════════════════

import os, re, json, time, pickle, threading
import numpy as np
import faiss
import pandas as pd
from tqdm import tqdm
from openai import OpenAI, RateLimitError, APIStatusError
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from Bio import Entrez
import requests

# ── ① Paste your Ollama Cloud API key here ──────────────────
OLLAMA_API_KEY = "92105380c829422c80f44eae16695182.Y82L4qllRikDSyslNzxPwE1B"   # ← REQUIRED

# ── ② Model — Gemma 4 31B Dense (Cloud Offloaded) ───────────
OLLAMA_MODEL = "gemma4:31b-cloud"

# ── Ollama Cloud client (OpenAI-compatible) ─────────────────
client = OpenAI(
    base_url="https://ollama.com/v1",  # ← FIXED: Must be /v1 for the OpenAI client
    api_key=OLLAMA_API_KEY,
)

# ── Rate-limit config ────────────────────────────────────────
# Ollama Cloud free tier: ~60 requests/min, 10k tokens/min
# We stay well under by capping and sleeping between calls.
_CALL_LOCK      = threading.Lock()
_last_call_ts   = 0.0
MIN_CALL_GAP    = 1.2   # seconds between API calls (≤50 req/min)
MAX_RETRIES     = 5
BASE_BACKOFF    = 2.0   # seconds (doubles on each retry)

# ── Token budget tracker ─────────────────────────────────────
_token_budget = {"prompt": 0, "completion": 0, "calls": 0}

def _update_budget(usage):
    if usage:
        _token_budget["prompt"]     += getattr(usage, "prompt_tokens",     0) or 0
        _token_budget["completion"] += getattr(usage, "completion_tokens", 0) or 0
    _token_budget["calls"] += 1

def show_budget():
    print(f"  API calls : {_token_budget['calls']}")
    print(f"  Prompt tok: {_token_budget['prompt']:,}")
    print(f"  Compl  tok: {_token_budget['completion']:,}")
    print(f"  Total  tok: {_token_budget['prompt'] + _token_budget['completion']:,}")


def chat(prompt: str,
         max_tokens: int = 512,
         temperature: float = 0.3,
         system: str | None = None) -> str:
    """
    Send a prompt to Ollama Cloud and return the reply text.

    • Enforces MIN_CALL_GAP between successive calls (rate-limit guard).
    • Retries on 429 / 503 with exponential back-off up to MAX_RETRIES.
    • Tracks token usage in _token_budget.
    """
    global _last_call_ts

    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    for attempt in range(MAX_RETRIES):
        # Enforce minimum gap between calls
        with _CALL_LOCK:
            wait = MIN_CALL_GAP - (time.time() - _last_call_ts)
            if wait > 0:
                time.sleep(wait)
            _last_call_ts = time.time()

        try:
            resp = client.chat.completions.create(
                model       = OLLAMA_MODEL,
                messages    = messages,
                max_tokens  = max_tokens,
                temperature = temperature,
            )
            _update_budget(resp.usage)
            return resp.choices[0].message.content.strip()

        except RateLimitError:
            backoff = BASE_BACKOFF * (2 ** attempt)
            print(f"  ⚠️  Rate limit hit — waiting {backoff:.0f}s (attempt {attempt+1}/{MAX_RETRIES})")
            time.sleep(backoff)

        except APIStatusError as e:
            if e.status_code in (503, 529):          # server overloaded
                backoff = BASE_BACKOFF * (2 ** attempt)
                print(f"  ⚠️  Server busy ({e.status_code}) — waiting {backoff:.0f}s")
                time.sleep(backoff)
            else:
                print(f"  ❌ API error {e.status_code}: {e.message}")
                return ""

        except Exception as e:
            print(f"  ❌ Unexpected error: {e}")
            return ""

    print(f"  ❌ All {MAX_RETRIES} retries exhausted — returning empty string")
    return ""


# ── Verify the key works ─────────────────────────────────────
for d in ["index", "data"]:
    os.makedirs(d, exist_ok=True)

print(f"Testing Ollama Cloud API with model: {OLLAMA_MODEL} …")
try:
    reply = chat("Reply with exactly two words: API OK", max_tokens=10)
    if reply:
        print(f"✅ API working — model replied: {reply}")
    else:
        print("⚠️  Empty reply — double-check your API key and model name.")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Check: key is correct, model name is valid, internet is on.")

Testing Ollama Cloud API with model: gemma4:31b-cloud …
✅ API working — model replied: API OK


## Cell 3 — Embeddings & PubMed Setup

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 3 — Local embeddings + PubMed config
# (No API key needed for either of these)
# ═══════════════════════════════════════════════════

# ── SentenceTransformer (runs locally, free) ─────────────────
MODEL_NAME    = "all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

print("Loading SentenceTransformer (local, no API) …")
st_model = SentenceTransformer(MODEL_NAME)
print("✅ Embeddings ready")

# ── PubMed / Entrez ──────────────────────────────────────────
Entrez.email = "your-email@example.com"   # ← any valid email
NCBI_API_KEY = ""                          # optional — ncbi.nlm.nih.gov/account
if NCBI_API_KEY:
    Entrez.api_key = NCBI_API_KEY
    print("✅ NCBI key set (10 req/s limit)")
else:
    print("ℹ️  No NCBI key — using public limit (3 req/s). Fine for this notebook.")


Loading SentenceTransformer (local, no API) …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embeddings ready
ℹ️  No NCBI key — using public limit (3 req/s). Fine for this notebook.


## Cell 4 — Load PubHealth Dataset

```bash
# Run once to download:
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/train.tsv
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/dev.tsv
```


In [ ]:
from google.colab import files
uploaded = files.upload()   # click and select train.tsv and test.tsv

Saving dev.tsv to dev.tsv
Saving test.tsv to test.tsv
Saving train.tsv to train.tsv


In [ ]:
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/train.tsv
!wget -q https://raw.githubusercontent.com/neemakot/Health-Fact-Checking/master/data/PUBHEALTH/dev.tsv


In [ ]:
# ═══════════════════════════════════════════════════
# CELL 4 — Load & preprocess PubHealth
# ═══════════════════════════════════════════════════

FALLBACK_DATA = [
    {"claim": "Vaccines cause autism.", "label": "false",
     "explanation": "Multiple large studies found no link between vaccines and autism."},
    {"claim": "Drinking water prevents kidney stones.", "label": "true",
     "explanation": "Adequate hydration reduces kidney stone formation risk."},
    {"claim": "COVID-19 vaccines contain microchips.", "label": "false",
     "explanation": "No evidence of microchips in any COVID-19 vaccine."},
    {"claim": "Exercise reduces risk of heart disease.", "label": "true",
     "explanation": "Regular physical activity strengthens the heart."},
    {"claim": "Vitamin C cures the common cold.", "label": "uncertain",
     "explanation": "Some studies show modest benefit but evidence is mixed."},
    {"claim": "Eating red meat causes cancer.", "label": "uncertain",
     "explanation": "Processed meat is classified as Group 1 carcinogen; unprocessed red meat as Group 2A."},
]

def load_pubhealth(path):
    df = pd.read_csv(path, sep="	", on_bad_lines="skip", engine="python")
    available = [c for c in ["claim", "label", "explanation"] if c in df.columns]
    df = df[available].dropna()
    if "explanation" not in df.columns:
        df["explanation"] = ""
    label_map = {"true": "true", "false": "false",
                 "mixture": "uncertain", "unproven": "uncertain"}
    df["label"] = df["label"].str.lower().map(label_map)
    df = df.dropna(subset=["label"])
    df["text"] = df["claim"].str.strip() + " " + df["explanation"].str.strip()
    return df[["claim", "label", "explanation", "text"]]

print("Loading PubHealth dataset …")
try:
    train_df = load_pubhealth("train.tsv")
    test_df  = load_pubhealth("dev.tsv")
    print(f"  Train: {len(train_df):,} | Dev: {len(test_df):,}")
except FileNotFoundError:
    print("  ⚠️  TSV files not found — using fallback sample (run the wget cell above).")
    for r in FALLBACK_DATA:
        r["text"] = r["claim"] + " " + r["explanation"]
    train_df = pd.DataFrame(FALLBACK_DATA)
    test_df  = train_df.copy()
print("✅ Dataset loaded")


Loading PubHealth dataset …
  Train: 6,657 | Dev: 1,214
✅ Dataset loaded


## Cell 5 — Build FAISS Index

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 5 — Build FAISS index from PubHealth corpus
# (local, uses SentenceTransformer — no API calls)
# ═══════════════════════════════════════════════════

corpus_records = train_df[["claim", "label", "text"]].to_dict("records")

print(f"Encoding {len(corpus_records):,} passages …")
corpus_texts = [r["text"] for r in corpus_records]
embeddings   = st_model.encode(
    corpus_texts, batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
).astype(np.float32)

index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(embeddings)
faiss.write_index(index, "index/pubhealth.bin")
with open("index/pubhealth_meta.pkl", "wb") as f:
    pickle.dump(corpus_records, f)

print(f"✅ FAISS index ready — {index.ntotal:,} vectors")


Encoding 6,657 passages …


Batches:   0%|          | 0/105 [00:00<?, ?it/s]

✅ FAISS index ready — 6,657 vectors


## Cell 6 — E3: Evidence Retrieval (FAISS + Live PubMed)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 6 — E3: Hybrid retrieval
# ═══════════════════════════════════════════════════

def search_pubmed(query, max_results=5):
    """Fetch live abstracts from PubMed (free NCBI Entrez API)."""
    results = []
    try:
        handle = Entrez.esearch(db="pubmed", term=query,
                                retmax=max_results, sort="relevance")
        ids    = Entrez.read(handle).get("IdList", [])
        handle.close()
        if not ids:
            return results

        handle  = Entrez.efetch(db="pubmed", id=",".join(ids),
                                rettype="abstract", retmode="xml")
        records = Entrez.read(handle)
        handle.close()

        for art in records.get("PubmedArticle", []):
            try:
                ml   = art["MedlineCitation"]
                data = ml["Article"]
                title = str(data.get("ArticleTitle", ""))
                abs_texts = data.get("Abstract", {}).get("AbstractText", [])
                abstract  = " ".join(str(a) for a in abs_texts)
                if len(abstract) > 50:
                    results.append({
                        "id":     str(ml.get("PMID", "")),
                        "title":  title,
                        "text":   f"{title}. {abstract[:500]}",
                        "source": "PubMed",
                        "label":  "pubmed_article",
                    })
            except Exception:
                continue
        time.sleep(0.35)   # respect NCBI 3 req/s public limit
    except Exception as e:
        print(f"  PubMed warning: {e}")
    return results


def retrieve_faiss(claim, top_k=5):
    """Top-k nearest passages from local FAISS index."""
    q      = st_model.encode([claim], convert_to_numpy=True,
                              show_progress_bar=False).astype(np.float32)
    D, I   = index.search(q, k=top_k)
    return [
        {**corpus_records[i], "rank": r+1,
         "distance": float(d), "source": "PubHealth"}
        for r, (i, d) in enumerate(zip(I[0], D[0])) if i != -1
    ]


def retrieve_all(claim, faiss_k=5, pubmed_k=3):
    """E3 — merged FAISS + live PubMed, deduplicated."""
    combined = retrieve_faiss(claim, faiss_k) +                search_pubmed(f"{claim} clinical evidence", pubmed_k)
    seen, out = set(), []
    for p in combined:
        key = p["text"][:80]
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


print("✅ E3 retrieval ready")


✅ E3 retrieval ready


## Cell 7 — Evidence Classification (Ollama Cloud)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 7 — Evidence classifier
# One API call classifies all passages at once.
# ═══════════════════════════════════════════════════

def classify_evidence(claim, passages):
    """
    Classify each passage as SUPPORT / OPPOSE / NEUTRAL in a single API call.
    Returns (supporting_list, opposing_list).
    """
    if not passages:
        return [], []

    # FIXED: Replaced the broken literal newline with the standard "\n" escape character.
    numbered = "\n".join(
        f'{i+1}. "{p["text"][:250]}"'
        for i, p in enumerate(passages)
    )

    prompt = f"""You are a medical evidence classifier. Your job is to decide whether each passage
SUPPORTS, OPPOSES, or is NEUTRAL toward the given claim.

Claim: "{claim}"

Passages:
{numbered}

Instructions:
- Respond with ONLY a numbered list, one line per passage.
- Use exactly this format: <number>: SUPPORT  or  <number>: OPPOSE  or  <number>: NEUTRAL
- Do NOT write anything else — no explanations, no headers.

Your answer:"""

    raw = chat(prompt, max_tokens=len(passages) * 15 + 30, temperature=0.1)

    pos_ev, neg_ev = [], []
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            # FIXED: Added replace to handle LLMs outputting "1. SUPPORT" instead of "1: SUPPORT"
            clean_line = line.replace(".", ":")
            num_s, label = clean_line.split(":", 1)
            idx   = int(num_s.strip()) - 1
            label = label.strip().upper()

            if 0 <= idx < len(passages):
                p = passages[idx]
                if "SUPPORT" in label:
                    pos_ev.append({**p, "ev_label": "SUPPORT"})
                elif "OPPOSE" in label:
                    neg_ev.append({**p, "ev_label": "OPPOSE"})
        except Exception:
            continue

    return pos_ev, neg_ev

print("✅ Evidence classifier ready")

✅ Evidence classifier ready


## Cell 8 — E2: Multi-Round Debate Agents (Ollama Cloud)

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 8 — E2: Pro / Con debate agents, 2 rounds
# ═══════════════════════════════════════════════════

SYS_PRO = "You are a rigorous medical debate agent. Your role is to argue IN FAVOUR of the claim using only the provided evidence. Be concise, cite evidence by number, and end with a clear conclusion."
SYS_CON = "You are a rigorous medical debate agent. Your role is to argue AGAINST the claim using only the provided evidence. Be concise, cite evidence by number, and end with a clear conclusion."
SYS_JDG = "You are an impartial medical fact-checking judge. Read both sides carefully and deliver a balanced, evidence-based verdict."


def _ev_block(ev_list, n=4):
    # FIXED: Replaced literal line breaks with "\n\n" and "\n"
    return "\n\n".join(
        f"[Evidence {i+1} | {e.get('source','')}]\n{e['text'][:300]}"
        for i, e in enumerate(ev_list[:n])
    )


def pro_agent_round1(claim, pos_ev):
    if not pos_ev:
        return {"argument": "No supporting evidence found.", "score": 0.0}
    prompt = f"""Claim: "{claim}"

Evidence:
{_ev_block(pos_ev)}

Build the strongest supporting argument. Max 200 words.
"""
    arg = chat(prompt, max_tokens=300, system=SYS_PRO)
    return {"argument": arg or "No response.",
            "score": len(pos_ev) * 0.35 + min(len(arg or "") / 400, 1.0)}


def con_agent_round1(claim, neg_ev):
    if not neg_ev:
        return {"argument": "No opposing evidence found.", "score": 0.0}
    prompt = f"""Claim: "{claim}"

Evidence:
{_ev_block(neg_ev)}

Build the strongest opposing argument. Max 200 words.
"""
    arg = chat(prompt, max_tokens=300, system=SYS_CON)
    return {"argument": arg or "No response.",
            "score": len(neg_ev) * 0.35 + min(len(arg or "") / 400, 1.0)}


def pro_rebuttal(claim, pro_arg, con_arg):
    prompt = f"""Claim: "{claim}"

Your Round 1 argument:
{pro_arg["argument"][:400]}

Opponent's Round 1 argument:
{con_arg["argument"][:400]}

Write a focused rebuttal that directly addresses the opponent's points.
Max 150 words.
"""
    arg = chat(prompt, max_tokens=250, system=SYS_PRO)
    return {"argument": arg or "No response.",
            "score": pro_arg["score"] + 0.2}


def con_rebuttal(claim, pro_arg, con_arg):
    prompt = f"""Claim: "{claim}"

Your Round 1 argument:
{con_arg["argument"][:400]}

Opponent's Round 1 argument:
{pro_arg["argument"][:400]}

Write a focused rebuttal that directly addresses the opponent's points.
Max 150 words.
"""
    arg = chat(prompt, max_tokens=250, system=SYS_CON)
    return {"argument": arg or "No response.",
            "score": con_arg["score"] + 0.2}


print("✅ E2 debate agents ready")

✅ E2 debate agents ready


## Cell 9 — E4: Uncertainty Quantification

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 9 — E4: Uncertainty scoring
# ═══════════════════════════════════════════════════

def quantify_uncertainty(result, pos_ev, neg_ev, all_passages):
    report, total = {}, 0.0

    # 1. Evidence conflict — both sides present
    c = 0.0
    if pos_ev and neg_ev:
        c = round(min(len(pos_ev), len(neg_ev)) /
                  max(len(pos_ev), len(neg_ev)) * 0.4, 3)
    report["evidence_conflict"] = {
        "score": c,
        "detail": f"{len(pos_ev)} supporting vs {len(neg_ev)} opposing"
    }
    total += c

    # 2. Evidence scarcity
    n = len(all_passages)
    s = 0.4 if n == 0 else (round((3 - n) / 3 * 0.3, 3) if n < 3 else 0.0)
    report["evidence_scarcity"] = {
        "score": s,
        "detail": f"{n} passage{'s' if n != 1 else ''} retrieved"
    }
    total += s

    # 3. Score proximity
    ps, cs  = result.get("pro_score", 0), result.get("con_score", 0)
    diff    = abs(ps - cs) / (ps + cs + 1e-6)
    p_score = round((0.15 - diff) / 0.15 * 0.2, 3) if diff < 0.15 else 0.0
    report["score_proximity"] = {
        "score": p_score,
        "detail": f"Pro={ps:.2f}  Con={cs:.2f}  (diff ratio={diff:.2f})"
    }
    total += p_score

    # 4. Source diversity
    srcs = {p.get("source","") for p in all_passages}
    d    = 0.1 if len(srcs) <= 1 and all_passages else 0.0
    report["source_diversity"] = {
        "score": d,
        "detail": f"Sources: {list(srcs)}"
    }
    total += d

    # 5. Claim ambiguity — one API call (FIXED: Replaced literal newlines with \n)
    amb = 0.0
    resp = chat(
        f'Is this medical claim ambiguous, multi-part, or hard to verify?\n'
        f'Claim: "{result.get("claim", "")}"\n'
        f'Reply with ONE word: YES or NO.',
        max_tokens=5, temperature=0.0
    ).upper()

    if "YES" in resp:
        amb = 0.15
    report["claim_ambiguity"] = {
        "score": amb,
        "detail": "Model flagged as ambiguous" if amb else "Claim appears unambiguous"
    }
    total += amb

    total = min(total, 1.0)
    pct   = round(total * 100, 1)
    level = "LOW" if pct < 20 else ("MEDIUM" if pct < 50 else "HIGH")

    return {
        "uncertainty_score":   pct,
        "uncertainty_level":   level,
        "uncertainty_sources": report,
        "interpretation": (
            f"Uncertainty {level} ({pct}%). " + {
                "LOW":    "Verdict is reliable.",
                "MEDIUM": "Treat verdict with caution.",
                "HIGH":   "Verdict unreliable — conflicting or scarce evidence.",
            }[level]
        )
    }

print("✅ E4 uncertainty quantification ready")

✅ E4 uncertainty quantification ready


## Cell 10 — Judge Agent

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 10 — Judge agent
# Reads all 4 arguments and delivers a structured verdict.
# ═══════════════════════════════════════════════════

def judge_agent(claim, pro_r1, con_r1, pro_r2, con_r2):
    prompt = f"""Claim: "{claim}"

=== ROUND 1 ===
PRO argued: {pro_r1["argument"][:300]}

CON argued: {con_r1["argument"][:300]}

=== ROUND 2 — REBUTTALS ===
PRO rebuttal: {pro_r2["argument"][:300]}

CON rebuttal: {con_r2["argument"][:300]}

Evaluate both rounds carefully. Respond in EXACTLY this format (no extra text):
VERDICT: <TRUE or FALSE or UNCERTAIN>
CONFIDENCE: <integer 0-100>
REASONING: <3-4 sentences explaining your verdict based on the debate>
"""
    raw     = chat(prompt, max_tokens=350, temperature=0.2, system=SYS_JDG)
    verdict, confidence, reasoning = "Uncertain", 50.0, ""

    for line in (raw or "").splitlines():
        line = line.strip()
        if line.upper().startswith("VERDICT:"):
            v = line.split(":", 1)[1].strip().upper()
            verdict = "True" if "TRUE" in v else ("False" if "FALSE" in v else "Uncertain")
        elif line.upper().startswith("CONFIDENCE:"):
            try:
                confidence = float(re.findall(r"[0-9.]+", line)[0])
            except Exception:
                pass
        elif line.upper().startswith("REASONING:"):
            reasoning = line.split(":", 1)[1].strip()

    ps = (pro_r1["score"] + pro_r2["score"]) / 2
    cs = (con_r1["score"] + con_r2["score"]) / 2

    # Fallback if model gave no structured output
    if not reasoning:
        verdict    = "True" if ps > cs else ("False" if cs > ps else "Uncertain")
        confidence = round(max(ps, cs) / (ps + cs + 1e-6) * 100, 1)
        reasoning  = "Score-based fallback (model returned unstructured output)."

    return {
        "claim":        claim,
        "verdict":      verdict,
        "confidence":   confidence,
        "reason":       reasoning,
        "pro_score":    round(ps, 3),
        "con_score":    round(cs, 3),
        "pro_r1":       pro_r1["argument"],
        "con_r1":       con_r1["argument"],
        "pro_rebuttal": pro_r2["argument"],
        "con_rebuttal": con_r2["argument"],
    }

print("✅ Judge agent ready")


✅ Judge agent ready


## Cell 11 — Full Pipeline

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 11 — verify_claim() + print_result()
# ═══════════════════════════════════════════════════

def verify_claim(claim, faiss_k=5, pubmed_k=3, verbose=True):
    """
    Full pipeline:
      1. Retrieve evidence (FAISS + PubMed)      [E3]
      2. Classify evidence                       [Ollama]
      3. Round 1 debate — Pro & Con              [E2]
      4. Round 2 rebuttals — Pro & Con           [E2]
      5. Judge verdict                           [Ollama]
      6. Uncertainty quantification              [E4]

    API calls per invocation: ~6 (classify×1, agents×4, judge×1, ambiguity×1 = 7 total)
    """
    if verbose:
        # FIXED: Replaced literal newlines with \n
        print(f"\n{'='*62}")
        print(f"  CLAIM: {claim}")
        print(f"{'='*62}")

    step = lambda s: print(f"  [{s}]") if verbose else None

    step("1/6  Retrieving evidence …")
    passages = retrieve_all(claim, faiss_k=faiss_k, pubmed_k=pubmed_k)
    if verbose:
        pm = sum(1 for p in passages if p.get("source") == "PubMed")
        print(f"       {len(passages)} passages  ({pm} live from PubMed)")

    step("2/6  Classifying evidence …")
    pos_ev, neg_ev = classify_evidence(claim, passages)
    if verbose:
        print(f"       {len(pos_ev)} supporting  |  {len(neg_ev)} opposing")

    step("3/6  Round 1 debate …")
    pro_r1 = pro_agent_round1(claim, pos_ev)
    con_r1 = con_agent_round1(claim, neg_ev)

    step("4/6  Round 2 rebuttals …")
    pro_r2 = pro_rebuttal(claim, pro_r1, con_r1)
    con_r2 = con_rebuttal(claim, pro_r1, con_r1)

    step("5/6  Judge evaluating …")
    result = judge_agent(claim, pro_r1, con_r1, pro_r2, con_r2)
    result["retrieved_count"]    = len(passages)
    result["pro_evidence_count"] = len(pos_ev)
    result["con_evidence_count"] = len(neg_ev)

    step("6/6  Quantifying uncertainty …")
    result.update(quantify_uncertainty(result, pos_ev, neg_ev, passages))

    return result


def print_result(r):
    icon   = {"True": "✅", "False": "❌", "Uncertain": "⚠️"}.get(r["verdict"], "⚠️")
    u_icon = {"LOW": "🟢", "MEDIUM": "🟡", "HIGH": "🔴"}.get(r["uncertainty_level"], "⚪")

    # FIXED: Replaced literal newlines with \n throughout the print statements
    print(f"\n{'─'*62}")
    print(f"  CLAIM       : {r['claim']}")
    print(f"  VERDICT     : {icon} {r['verdict']}   (Confidence: {r['confidence']}%)")
    print(f"  UNCERTAINTY : {u_icon} {r['uncertainty_level']} ({r['uncertainty_score']}%)")
    print(f"  REASONING   : {r['reason'][:300]}")
    print(f"  INTERPRET   : {r['interpretation']}")

    print(f"\n  ── Evidence ──────────────────────────────────────────")
    print(f"  Retrieved   : {r['retrieved_count']}")
    print(f"  Supporting  : {r['pro_evidence_count']}")
    print(f"  Opposing    : {r['con_evidence_count']}")

    print(f"\n  ── Uncertainty Breakdown ─────────────────────────────")
    for k, v in r["uncertainty_sources"].items():
        bar = "█" * int(v["score"] * 20)
        print(f"  {k:<22}: {v['score']:.2f}  {bar:<8}  {v['detail']}")

    print(f"\n  ── Debate ────────────────────────────────────────────")
    print(f"  PRO  R1: {r.get('pro_r1', 'N/A')[:100]} …")
    print(f"  CON  R1: {r.get('con_r1', 'N/A')[:100]} …")
    print(f"  PRO  R2: {r.get('pro_rebuttal', 'N/A')[:100]} …")
    print(f"  CON  R2: {r.get('con_rebuttal', 'N/A')[:100]} …")
    print(f"{'─'*62}")


print("✅ Full pipeline ready")

✅ Full pipeline ready


## Cell 12 — Test on Sample Claims

In [ ]:
SAMPLE_CLAIMS = [
    "Vaccines cause autism.",
    "Exercise reduces the risk of heart disease.",
    "Vitamin C cures the common cold.",
    "COVID-19 vaccines are safe and effective.",
]

print("🔬 Running verification on sample claims …")
for claim in SAMPLE_CLAIMS:
    result = verify_claim(claim, faiss_k=5, pubmed_k=3, verbose=True)
    print_result(result)

print("── API usage so far ──")
show_budget()


🔬 Running verification on sample claims …

  CLAIM: Vaccines cause autism.
  [1/6  Retrieving evidence …]
       7 passages  (2 live from PubMed)
  [2/6  Classifying evidence …]
       1 supporting  |  3 opposing
  [3/6  Round 1 debate …]
  [4/6  Round 2 rebuttals …]
  [5/6  Judge evaluating …]
  [6/6  Quantifying uncertainty …]

──────────────────────────────────────────────────────────────
  CLAIM       : Vaccines cause autism.
  VERDICT     : ❌ False   (Confidence: 100.0%)
  UNCERTAINTY : 🟢 LOW (13.3%)
  REASONING   : The PRO argument relies on a widely debunked misinterpretation of a 2004 CDC study, which analyzed the timing of vaccination rather than a causal link to autism. Extensive global epidemiological studies and systematic reviews have consistently found no evidence that vaccines cause autism. The CON ar
  INTERPRET   : Uncertainty LOW (13.3%). Verdict is reliable.

  ── Evidence ──────────────────────────────────────────
  Retrieved   : 7
  Supporting  : 1
  Opposing    : 

## Cell 13 — Evaluation on PubHealth Dev Set

In [ ]:
# ═══════════════════════════════════════════════════
# CELL 13 — Batch evaluation
# Each claim uses ~7 API calls → 20 claims ≈ 140 calls
# With MIN_CALL_GAP=1.2s that takes ~3 min (well within limits)
# ═══════════════════════════════════════════════════

EVAL_SAMPLE_SIZE = 20

print(f"Evaluating on {EVAL_SAMPLE_SIZE} dev claims …")
sample = test_df.sample(min(EVAL_SAMPLE_SIZE, len(test_df)),
                        random_state=42).reset_index(drop=True)

true_labels, pred_labels, confidences, unc_scores = [], [], [], []

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Verifying"):
    try:
        r = verify_claim(row["claim"], faiss_k=5, pubmed_k=2, verbose=False)
        pred_labels.append(r["verdict"].lower())
        confidences.append(r["confidence"])
        unc_scores.append(r["uncertainty_score"])
        true_labels.append(row["label"])
    except Exception as e:
        # FIXED: Replaced literal newline with \n
        print(f"\n  Error: {e}")
        pred_labels.append("uncertain")
        confidences.append(50.0)
        unc_scores.append(50.0)
        true_labels.append(row["label"])

LABELS = ["true", "false", "uncertain"]
acc = accuracy_score(true_labels, pred_labels) * 100
f1  = f1_score(true_labels, pred_labels, average="macro",
               labels=LABELS, zero_division=0) * 100

# FIXED: Replaced literal newlines with \n throughout the print statements
print(f"\n{'System':<50} {'Acc%':>6} {'F1%':>6}")
print("─" * 63)
print(f"{'Ollama Cloud (gemma3:27b) + PubMed + 2-Round + UQ':<50} {acc:>6.1f} {f1:>6.1f}")
print(f"\nMean Confidence : {np.mean(confidences):.1f}%")
print(f"Mean Uncertainty: {np.mean(unc_scores):.1f}%")
print(f"\nClassification Report:")
print(classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0))

print("\n── Final API usage ──")
show_budget()

Evaluating on 20 dev claims …


Verifying: 100%|██████████| 20/20 [24:43<00:00, 74.17s/it]


System                                               Acc%    F1%
───────────────────────────────────────────────────────────────
Ollama Cloud (gemma3:27b) + PubMed + 2-Round + UQ    40.0   37.4

Mean Confidence : 97.5%
Mean Uncertainty: 34.0%

Classification Report:
              precision    recall  f1-score   support

        true       0.50      0.10      0.17        10
       false       1.00      0.67      0.80         9
   uncertain       0.08      1.00      0.15         1

    accuracy                           0.40        20
   macro avg       0.53      0.59      0.37        20
weighted avg       0.70      0.40      0.45        20


── Final API usage ──
  API calls : 133
  Prompt tok: 29,242
  Compl  tok: 8,827
  Total  tok: 38,069


## Cell 14 — Interactive: Verify Any Claim

Change `YOUR_CLAIM` and run.


In [ ]:
YOUR_CLAIM = "Drinking coffee reduces the risk of Alzheimer's disease."

result = verify_claim(YOUR_CLAIM, faiss_k=5, pubmed_k=3, verbose=True)
print_result(result)
print("── API usage so far ──")
show_budget()



  CLAIM: Drinking coffee reduces the risk of Alzheimer's disease.
  [1/6  Retrieving evidence …]
       6 passages  (1 live from PubMed)
  [2/6  Classifying evidence …]
       2 supporting  |  0 opposing
  [3/6  Round 1 debate …]
  [4/6  Round 2 rebuttals …]
  [5/6  Judge evaluating …]
  [6/6  Quantifying uncertainty …]

──────────────────────────────────────────────────────────────
  CLAIM       : Drinking coffee reduces the risk of Alzheimer's disease.
  VERDICT     : ⚠️ Uncertain   (Confidence: 70.0%)
  UNCERTAINTY : 🟢 LOW (15.0%)
  REASONING   : The PRO side provides a specific study suggesting a positive association between coffee/caffeine and cognitive health, but fails to establish a definitive causal link. The CON side correctly identifies the distinction between correlation and causation, noting that confounding variables may influence
  INTERPRET   : Uncertainty LOW (15.0%). Verdict is reliable.

  ── Evidence ──────────────────────────────────────────
  Retrieved   : 6
  Su

## Cell 15 — Save Results to JSON

In [ ]:
output_path = "data/verification_results.json"
to_save     = []

for claim in SAMPLE_CLAIMS:
    try:
        r = verify_claim(claim, faiss_k=5, pubmed_k=2, verbose=False)
        to_save.append({k: v for k, v in r.items()
                        if k not in ["pro_r1","con_r1","pro_rebuttal","con_rebuttal"]})
    except Exception as e:
        print(f"  Error: {e}")

with open(output_path, "w") as f:
    json.dump(to_save, f, indent=2)

print(f"✅ Saved {len(to_save)} results → {output_path}")
print("✅ ALL CELLS COMPLETE")
print("  E2 (2-round debate)            ✅")
print("  E3 (live PubMed)               ✅")
print("  E4 (uncertainty quantification)✅")
print(f"  LLM: {OLLAMA_MODEL} via Ollama Cloud  ✅")

print("── Total API usage ──")
show_budget()


✅ Saved 4 results → data/verification_results.json
✅ ALL CELLS COMPLETE
  E2 (2-round debate)            ✅
  E3 (live PubMed)               ✅
  E4 (uncertainty quantification)✅
  LLM: gemma4:31b-cloud via Ollama Cloud  ✅
── Total API usage ──
  API calls : 166
  Prompt tok: 37,401
  Compl  tok: 11,748
  Total  tok: 49,149
